> 在线讲义：[第 14 章 · 从单次请求到持续到达的负载：什么时候来，比来多少更要紧](https://wikieden.github.io/ai-infra-from-constraints/lessons/14-workload-arrival.html) · 本书《从约束推导 AI Infra》

# 第 14 章 · 从单次请求到持续到达的负载：什么时候来，比来多少更要紧

配套讲义：仓库里 `ai-infra/notes/lessons/14-workload-arrival.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 三个指标 | 时间差 | `request_metrics(0, 0.8, 5.8, 101)` | 0.8 s、5.8 s、50 ms |
| 第二步 p95 | 最近秩 | `p95_rank(4)`、`p95_rank(480)` | 第 4 个、第 456 个 |
| 第三步 阶段需求 | 两类请求按比例加 | `window_demand(4 * 0.5)` 等 | 18,432 / 4,604；29,900.8 / 1,736.8；6,963.2 / 7,471.2 |
| 第三步 decode 处理率 | 读取量 ÷ 带宽 | `decode_rate()` | 一步 22.89 ms，单卡约 2,796 步/秒 |
| 第三步 积压 | 逐秒递推 | `simulate_backlog([...], mu)` | 峰值约 112,792 步，排空约 20.17 s |

例题 3-1（两类请求、RTX PRO 6000 1.792 TB/s、64 条序列、Qwen3-8B）取自 AII 第 3 章 §3.1；t_a = 0、t_1 = 0.8、t_G = 5.8 的请求是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
KV = 147_456  # Qwen3-8B 每 token 的 KV（字节）
R_W = 15_136_811_008  # 一步 decode 的共享权重读取（字节）

**函数 `request_metrics`**：(TTFT, 完整请求时间, 平均 token 间隔)，单位秒。

In [ ]:
def request_metrics(t_a: float, t_1: float, t_g: float, G: int) -> tuple[float, float, float]:
    """(TTFT, 完整请求时间, 平均 token 间隔)，单位秒。"""
    return t_1 - t_a, t_g - t_a, (t_g - t_1) / (G - 1)

**函数 `p95_rank`**：最近秩法：n 个观测里 p95 是排序后的第几个。

In [ ]:
def p95_rank(n: int) -> int:
    """最近秩法：n 个观测里 p95 是排序后的第几个。"""
    return math.ceil(0.95 * n)

**函数 `state_occupancy`**：稳定时等待任务的平均状态占用 ≈ λ · M0 · τ。

In [ ]:
def state_occupancy(lam: float, m0: float, tau: float) -> float:
    """稳定时等待任务的平均状态占用 ≈ λ · M0 · τ。"""
    return lam * m0 * tau

**函数 `window_demand`**：(新输入 token/s, 后续 decode 步/s)。长输入类 8192 入 256 出；长输出类 1024 入 2048 出。

In [ ]:
def window_demand(rate_long_in: float, rate_long_out: float) -> tuple[float, float]:
    """(新输入 token/s, 后续 decode 步/s)。长输入类 8192 入 256 出；长输出类 1024 入 2048 出。"""
    return rate_long_in * 8192 + rate_long_out * 1024, rate_long_in * 255 + rate_long_out * 2047

**函数 `decode_rate`**：单卡 decode 上限：(一步毫秒, 每秒步数)。一步读权重 + 每条序列旧 KV，再写一份新 KV。

In [ ]:
def decode_rate(seqs: int = 64, avg_ctx: int = 2742, beta: float = 1.792e12) -> tuple[float, float]:
    """单卡 decode 上限：(一步毫秒, 每秒步数)。一步读权重 + 每条序列旧 KV，再写一份新 KV。"""
    step_bytes = R_W + seqs * avg_ctx * KV + seqs * KV
    t = step_bytes / beta
    return t * 1e3, seqs / t

**函数 `simulate_backlog`**：每个元素是一秒内的到达速率；返回 (积压峰值, 排空时刻)。之后补 60 秒不再到达。

In [ ]:
def simulate_backlog(arrivals: list[float], mu: float, dt: float = 0.01) -> tuple[float, float | None]:
    """每个元素是一秒内的到达速率；返回 (积压峰值, 排空时刻)。之后补 60 秒不再到达。"""
    q, peak, t, t_empty = 0.0, 0.0, 0.0, None
    for a in arrivals + [0.0] * 60:
        for _ in range(int(round(1 / dt))):
            q = max(0.0, q + (a - mu) * dt)  # 到达 − 处理，不能为负
            t += dt
            peak = max(peak, q)
            if q == 0 and peak > 0 and t_empty is None:
                t_empty = t
    return peak, t_empty

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步：一个请求的三个指标（编的时间点）")
ttft, req, tok = request_metrics(0, 0.8, 5.8, 101)
print(f"  TTFT {ttft:.1f} s，完整 {req:.1f} s，平均 token 间隔 {tok * 1e3:.0f} ms")
print(f"  p95：4 条取第 {p95_rank(4)} 个，480 条取第 {p95_rank(480)} 个")
print(f"  1 GiB 占 10 秒 = {1 * 10} GiB·s")

In [ ]:
print("\n第三步：例题 3-1 的阶段需求（每秒 4 条请求）")
for name, (a, b) in (("均匀混合", (2, 2)), ("前一分钟 9:1", (3.6, 0.4)), ("后一分钟 1:9", (0.4, 3.6))):
    inp, dec = window_demand(a, b)
    print(f"  {name}：输入 {inp:,.1f} token/s，decode {dec:,.1f} 步/s")
ms, rate = decode_rate()
print(f"  单卡一步 {ms:.2f} ms → {rate:,.2f} 步/s；两张 {2 * rate:,.2f} 步/s")
mu = 2 * rate
arrivals = [4604.0] * 60 + [1736.8] * 60 + [7471.2] * 60
peak, t_empty = simulate_backlog(arrivals[60:], mu)
print(f"  时段变化：积压峰值 {peak:,.0f} 步，到达停止后 {t_empty - 120:.2f} s 排空")

## 3. 代码里才看得到的

1. 处理率是用均匀混合时的平均上下文 2742（精确值 2741.8）算的，代码得 2,795.70 步/秒、积压 112,788 步，原书是 2,795.66 和 112,792，差在舍入，测试放宽到几十步以内；原书说三个时间窗都沿用这个数，严格说后一分钟长输出类多、平均上下文更短，真实处理率会略高。
2. `simulate_backlog` 用 0.01 秒的小步长逐秒推进，和"按 60 秒窗口一步递推"结果一样，因为每个窗口内到达和处理都是匀速。
3. `p95_rank` 用 `math.ceil`；不同工具的百分位定义不一样（插值法会给出非整数位置），比较时先确认口径。

## 4. 自己改一改

- 把后一分钟的比例从 1:9 改成 3:7，看还会不会积压。
- 把 decode 卡数从 2 改成 3，重跑 `simulate_backlog`。
- 用 `state_occupancy(λ, M0, τ)` 算一算：工具等待从 10 秒延长到 60 秒，平均占用涨几倍。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 备菜一大堆，出菜一盘盘；
> 下单、接单、送完，三个钟各量一段；
> 闲时的力气存不到忙时，超出乘时长就是积压；
> 座位坐满了，灶台空着也得等。